# QTIP vs NWC Paired t-test

This notebook follows the same JSON-loading style as `plot_bpp_acc_v5_csv.ipynb`.

- QTIP experiment: `QTIP_no_e2e`
- NWC experiment: `NWC_ql_ldlq128_rnorm_ft`
- Default target: Llama3-8B
- Pairing is fully manual: edit `MANUAL_KEY_PAIRS` in the config cell
- Common Sense: for each manual pair, run paired t-test across the 6 common-sense tasks
- MMLU: for each manual pair, run paired t-test across matched detailed MMLU subjects

Important:
- Each pair is defined by `(qtip_key, nwc_key)`
- Within each pair, values are matched by the same task name or the same MMLU subject name
- Therefore the number of p-values is exactly `len(MANUAL_KEY_PAIRS)` for Common Sense and also exactly `len(MANUAL_KEY_PAIRS)` for MMLU

In [1]:
from pathlib import Path
import glob
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import ttest_rel

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

ROOT = Path("/home/jgryu/workspace/weight_compression")

QTIP_LABEL = "QTIP_no_e2e"
NWC_LABEL = "NWC_ql_ldlq128_rnorm_ft"

QTIP_DIR = ROOT / "hf_model_comp_results/qtip/llama3_8b/ft1"
NWC_DIR = ROOT / "hf_model_comp_results/meta-llama--Meta-Llama-3-8B/ql_ldlq128_rnorm_ft"

# Edit this list directly.
# Format: (qtip_key, nwc_key)
MANUAL_KEY_PAIRS = [
    (2, 30),
    (3, 75),
    (4, 300),
    (5, 1000),
    (6, 10000),
]

COMMON_SENSE_TASKS = [
    "arc_challenge",
    "arc_easy",
    "boolq",
    "piqa",
    "winogrande",
    "hellaswag",
]

MMLU_GROUP_KEYS = {
    "mmlu",
    "mmlu_humanities",
    "mmlu_social_sciences",
    "mmlu_other",
    "mmlu_stem",
}

SAVE_CSV = True
OUTPUT_DIR = ROOT / "notebooks/exp_results_csv/paired_ttest_qtip_vs_nwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert QTIP_DIR.exists(), f"Missing directory: {QTIP_DIR}"
assert NWC_DIR.exists(), f"Missing directory: {NWC_DIR}"
assert MANUAL_KEY_PAIRS, "MANUAL_KEY_PAIRS is empty. Add at least one pair."

print(f"QTIP dir: {QTIP_DIR}")
print(f"NWC dir : {NWC_DIR}")
print(f"CSV out : {OUTPUT_DIR}")
print(f"Manual key pairs: {MANUAL_KEY_PAIRS}")


QTIP dir: /home/jgryu/workspace/weight_compression/hf_model_comp_results/qtip/llama3_8b/ft1
NWC dir : /home/jgryu/workspace/weight_compression/hf_model_comp_results/meta-llama--Meta-Llama-3-8B/ql_ldlq128_rnorm_ft
CSV out : /home/jgryu/workspace/weight_compression/notebooks/exp_results_csv/paired_ttest_qtip_vs_nwc
Manual key pairs: [(2, 30), (3, 75), (4, 300), (5, 1000), (6, 10000)]


In [2]:
def extract_key_from_filename(filename):
    m_lambda = re.search(r"lmbda(\d+(\.\d+)?)", filename)
    if m_lambda:
        val = float(m_lambda.group(1))
        return int(val) if val.is_integer() else val

    m_bit = re.search(r"(\d+(\.\d+)?)\s*bit", filename.lower())
    if m_bit:
        val = float(m_bit.group(1))
        return int(val) if val.is_integer() else val

    if "base" in filename.lower():
        return "base"

    return None


def normalize_key(key):
    if isinstance(key, str):
        stripped = key.strip()
        try:
            numeric = float(stripped)
            return int(numeric) if numeric.is_integer() else numeric
        except ValueError:
            return stripped.lower()
    return key


def get_acc(task_result):
    for key in ("acc,none", "acc"):
        if key in task_result and task_result[key] is not None:
            return task_result[key]
    return None


def get_bpp_from_result_json(file_path, result_dict):
    if result_dict.get("bpp_loss") is not None:
        return float(result_dict["bpp_loss"])
    if result_dict.get("bpp") is not None:
        return float(result_dict["bpp"])

    m_bit = re.search(r"(\d+(\.\d+)?)\s*bit", os.path.basename(file_path).lower())
    if m_bit:
        return float(m_bit.group(1))

    return np.nan


def load_experiment_results(exp_dir, exp_label):
    points = {}

    for file_path in sorted(glob.glob(str(exp_dir / "*.json"))):
        filename = os.path.basename(file_path)
        key = extract_key_from_filename(filename)
        if key is None:
            continue

        point = points.setdefault(
            key,
            {
                "Experiment": exp_label,
                "key": key,
                "key_norm": normalize_key(key),
                "bpp": np.nan,
                "source_files": [],
                "mmlu_subjects": {},
            },
        )

        with open(file_path, "r") as f:
            data = json.load(f)

        point["source_files"].append(filename)

        if "result" in filename and "zeroshot" not in filename and "mmlu" not in filename:
            bpp = get_bpp_from_result_json(file_path, data)
            if np.isfinite(bpp):
                point["bpp"] = bpp

        if "results" not in data:
            continue

        results = data["results"]

        for task in COMMON_SENSE_TASKS:
            if task in results:
                value = get_acc(results[task])
                if value is not None:
                    point[task] = value

        if "mmlu" in results:
            value = get_acc(results["mmlu"])
            if value is not None:
                point["mmlu"] = value

        for task_name, task_result in results.items():
            if not task_name.startswith("mmlu_"):
                continue
            if task_name in MMLU_GROUP_KEYS:
                continue

            value = get_acc(task_result)
            if value is not None:
                point["mmlu_subjects"][task_name] = value

        if not np.isfinite(point["bpp"]) and isinstance(key, (int, float)):
            point["bpp"] = float(key)

    df = pd.DataFrame(points.values()).sort_values("bpp").reset_index(drop=True)
    return df


def build_pairs_from_manual_key_pairs(qtip_df, nwc_df, manual_key_pairs):
    pair_rows = []

    normalized_pairs = [(normalize_key(qtip_key), normalize_key(nwc_key)) for qtip_key, nwc_key in manual_key_pairs]
    qtip_keys = [pair[0] for pair in normalized_pairs]
    nwc_keys = [pair[1] for pair in normalized_pairs]

    if len(set(qtip_keys)) != len(qtip_keys):
        raise ValueError(f"Duplicated QTIP keys in MANUAL_KEY_PAIRS: {manual_key_pairs}")
    if len(set(nwc_keys)) != len(nwc_keys):
        raise ValueError(f"Duplicated NWC keys in MANUAL_KEY_PAIRS: {manual_key_pairs}")

    for raw_qtip_key, raw_nwc_key in manual_key_pairs:
        qtip_key = normalize_key(raw_qtip_key)
        nwc_key = normalize_key(raw_nwc_key)

        qtip_match = qtip_df[qtip_df["key_norm"] == qtip_key]
        nwc_match = nwc_df[nwc_df["key_norm"] == nwc_key]

        if qtip_match.empty:
            raise KeyError(f"QTIP key {raw_qtip_key} not found. Available keys: {qtip_df['key'].tolist()}")
        if nwc_match.empty:
            raise KeyError(f"NWC key {raw_nwc_key} not found. Available keys: {nwc_df['key'].tolist()}")
        if len(qtip_match) != 1:
            raise ValueError(f"QTIP key {raw_qtip_key} matched multiple rows.")
        if len(nwc_match) != 1:
            raise ValueError(f"NWC key {raw_nwc_key} matched multiple rows.")

        qtip_row = qtip_match.iloc[0]
        nwc_row = nwc_match.iloc[0]

        pair_rows.append(
            {
                "qtip_idx": int(qtip_row.name),
                "nwc_idx": int(nwc_row.name),
                "qtip_key": qtip_row["key"],
                "nwc_key": nwc_row["key"],
                "qtip_bpp": float(qtip_row["bpp"]),
                "nwc_bpp": float(nwc_row["bpp"]),
                "abs_bpp_gap": abs(float(nwc_row["bpp"]) - float(qtip_row["bpp"])),
            }
        )

    return pd.DataFrame(pair_rows)


def build_common_sense_task_pairs(qtip_df, nwc_df, pairs_df, tasks=COMMON_SENSE_TASKS):
    rows = []

    for _, pair in pairs_df.iterrows():
        for task in tasks:
            qtip_score = qtip_df.loc[pair["qtip_idx"], task]
            nwc_score = nwc_df.loc[pair["nwc_idx"], task]

            if pd.isna(qtip_score) or pd.isna(nwc_score):
                continue

            rows.append(
                {
                    "task": task,
                    "qtip_key": pair["qtip_key"],
                    "nwc_key": pair["nwc_key"],
                    "qtip_bpp": pair["qtip_bpp"],
                    "nwc_bpp": pair["nwc_bpp"],
                    "abs_bpp_gap": pair["abs_bpp_gap"],
                    "qtip_score": qtip_score,
                    "nwc_score": nwc_score,
                    "diff_nwc_minus_qtip": nwc_score - qtip_score,
                }
            )

    return pd.DataFrame(rows)


def run_common_sense_tests_by_pair(common_task_pairs_df):
    rows = []

    for (qtip_key, nwc_key), group in common_task_pairs_df.groupby(["qtip_key", "nwc_key"]):
        if len(group) < 2:
            t_stat = np.nan
            p_value = np.nan
        else:
            stat = ttest_rel(group["nwc_score"], group["qtip_score"])
            t_stat = stat.statistic
            p_value = stat.pvalue

        rows.append(
            {
                "qtip_key": qtip_key,
                "nwc_key": nwc_key,
                "qtip_bpp": group["qtip_bpp"].iloc[0],
                "nwc_bpp": group["nwc_bpp"].iloc[0],
                "n_tasks": len(group),
                "qtip_mean": group["qtip_score"].mean(),
                "nwc_mean": group["nwc_score"].mean(),
                "mean_diff_nwc_minus_qtip": group["diff_nwc_minus_qtip"].mean(),
                "t_stat": t_stat,
                "p_value": p_value,
            }
        )

    return pd.DataFrame(rows).sort_values(["qtip_key", "nwc_key"]).reset_index(drop=True)


def build_mmlu_subject_pairs(qtip_df, nwc_df, pairs_df):
    rows = []

    for _, pair in pairs_df.iterrows():
        qtip_subjects = qtip_df.loc[pair["qtip_idx"], "mmlu_subjects"]
        nwc_subjects = nwc_df.loc[pair["nwc_idx"], "mmlu_subjects"]
        common_subjects = sorted(set(qtip_subjects) & set(nwc_subjects))

        for subject in common_subjects:
            qtip_score = qtip_subjects[subject]
            nwc_score = nwc_subjects[subject]
            rows.append(
                {
                    "subject": subject,
                    "qtip_key": pair["qtip_key"],
                    "nwc_key": pair["nwc_key"],
                    "qtip_bpp": pair["qtip_bpp"],
                    "nwc_bpp": pair["nwc_bpp"],
                    "abs_bpp_gap": pair["abs_bpp_gap"],
                    "qtip_score": qtip_score,
                    "nwc_score": nwc_score,
                    "diff_nwc_minus_qtip": nwc_score - qtip_score,
                }
            )

    return pd.DataFrame(rows)


def run_mmlu_tests_by_pair(mmlu_subject_pairs_df):
    rows = []

    for (qtip_key, nwc_key), group in mmlu_subject_pairs_df.groupby(["qtip_key", "nwc_key"]):
        if len(group) < 2:
            t_stat = np.nan
            p_value = np.nan
        else:
            stat = ttest_rel(group["nwc_score"], group["qtip_score"])
            t_stat = stat.statistic
            p_value = stat.pvalue

        rows.append(
            {
                "qtip_key": qtip_key,
                "nwc_key": nwc_key,
                "qtip_bpp": group["qtip_bpp"].iloc[0],
                "nwc_bpp": group["nwc_bpp"].iloc[0],
                "n_subjects": len(group),
                "qtip_mean": group["qtip_score"].mean(),
                "nwc_mean": group["nwc_score"].mean(),
                "mean_diff_nwc_minus_qtip": group["diff_nwc_minus_qtip"].mean(),
                "t_stat": t_stat,
                "p_value": p_value,
            }
        )

    return pd.DataFrame(rows).sort_values(["qtip_key", "nwc_key"]).reset_index(drop=True)


In [3]:
qtip_df = load_experiment_results(QTIP_DIR, QTIP_LABEL)
nwc_df = load_experiment_results(NWC_DIR, NWC_LABEL)
pairs_df = build_pairs_from_manual_key_pairs(qtip_df, nwc_df, MANUAL_KEY_PAIRS)

print("Available QTIP keys")
display(qtip_df[["key", "bpp"] + COMMON_SENSE_TASKS + ["mmlu"]])

print("Available NWC keys")
display(nwc_df[["key", "bpp"] + COMMON_SENSE_TASKS + ["mmlu"]])

print("Manual key mapping")
display(pairs_df)

if SAVE_CSV:
    qtip_df.to_csv(OUTPUT_DIR / "qtip_loaded_points.csv", index=False)
    nwc_df.to_csv(OUTPUT_DIR / "nwc_loaded_points.csv", index=False)
    pairs_df.to_csv(OUTPUT_DIR / "pair_mapping.csv", index=False)
    print(f"Saved pair mapping to {OUTPUT_DIR / 'pair_mapping.csv'}")


Available QTIP keys


,key,bpp,arc_challenge,arc_easy,boolq,piqa,winogrande,hellaswag,mmlu
0,2,2.0,0.431741,0.764731,0.774924,0.755169,0.697711,0.534157,0.502421
1,3,3.0,0.481229,0.795875,0.814373,0.791621,0.730071,0.585043,0.591155
2,4,4.0,0.501706,0.800926,0.815291,0.793254,0.734017,0.597789,0.615083
3,5,5.0,0.508532,0.804293,0.811927,0.793254,0.732439,0.598387,0.616080
4,6,6.0,0.508532,0.800505,0.813456,0.794342,0.741910,0.598785,0.615439
5,7,7.0,0.508532,0.803030,0.811621,0.793798,0.737964,0.598287,NaN


Available NWC keys


,key,bpp,arc_challenge,arc_easy,boolq,piqa,winogrande,hellaswag,mmlu
0,30,2.301533,0.405290,0.742424,0.801223,0.764962,0.704815,0.525891,0.496724
1,50,2.669362,0.458191,0.788721,0.805810,0.776931,0.726125,0.560944,0.551987
2,75,2.966646,0.480375,0.788300,0.820795,0.784548,0.726125,0.578670,0.585315
3,100,3.167211,0.470990,0.799242,0.813456,0.789445,0.737174,0.584445,0.601909
4,300,3.954827,0.505973,0.800926,0.808563,0.794342,0.748224,0.595698,0.618786
5,1000,4.777825,0.509386,0.800505,0.806422,0.798150,0.741910,0.598785,0.619855
6,10000,5.981024,0.504266,0.801768,0.817125,0.795430,0.731650,0.602071,0.621493


Manual key mapping


,qtip_idx,nwc_idx,qtip_key,nwc_key,qtip_bpp,nwc_bpp,abs_bpp_gap
0,0,0,2,30,2.0,2.301533,0.301533
1,1,2,3,75,3.0,2.966646,0.033354
2,2,4,4,300,4.0,3.954827,0.045173
3,3,5,5,1000,5.0,4.777825,0.222175
4,4,6,6,10000,6.0,5.981024,0.018976


Saved pair mapping to /home/jgryu/workspace/weight_compression/notebooks/exp_results_csv/paired_ttest_qtip_vs_nwc/pair_mapping.csv


In [4]:
common_task_pairs_df = build_common_sense_task_pairs(qtip_df, nwc_df, pairs_df)
common_by_pair_df = run_common_sense_tests_by_pair(common_task_pairs_df)

print(f"Common Sense p-value rows: {len(common_by_pair_df)}")
print(f"Expected rows from MANUAL_KEY_PAIRS: {len(MANUAL_KEY_PAIRS)}")

print("Common Sense paired t-test by manual key pair")
display(common_by_pair_df)

print("Matched Common Sense task-level values")
display(common_task_pairs_df)

if SAVE_CSV:
    common_by_pair_df.to_csv(OUTPUT_DIR / "common_sense_paired_ttest_by_pair.csv", index=False)
    common_task_pairs_df.to_csv(OUTPUT_DIR / "common_sense_task_pairs.csv", index=False)
    print(f"Saved Common Sense tables to {OUTPUT_DIR}")


Common Sense p-value rows: 5
Expected rows from MANUAL_KEY_PAIRS: 5
Common Sense paired t-test by manual key pair


,qtip_key,nwc_key,qtip_bpp,nwc_bpp,n_tasks,qtip_mean,nwc_mean,mean_diff_nwc_minus_qtip,t_stat,p_value
0,2.0,30.0,2.0,2.301533,6,0.659739,0.657434,-0.002304,-0.277292,0.792655
1,3.0,75.0,3.0,2.966646,6,0.699702,0.696469,-0.003233,-1.481658,0.198524
2,4.0,300.0,4.0,3.954827,6,0.707164,0.708954,0.001790,0.618193,0.563517
3,5.0,1000.0,5.0,4.777825,6,0.708139,0.709193,0.001054,0.467632,0.659721
4,6.0,10000.0,6.0,5.981024,6,0.709588,0.708718,-0.000870,-0.394395,0.709550


Matched Common Sense task-level values


,task,qtip_key,nwc_key,qtip_bpp,nwc_bpp,abs_bpp_gap,qtip_score,nwc_score,diff_nwc_minus_qtip
0,arc_challenge,2.0,30.0,2.0,2.301533,0.301533,0.431741,0.405290,-0.026451
1,arc_easy,2.0,30.0,2.0,2.301533,0.301533,0.764731,0.742424,-0.022306
2,boolq,2.0,30.0,2.0,2.301533,0.301533,0.774924,0.801223,0.026300
3,piqa,2.0,30.0,2.0,2.301533,0.301533,0.755169,0.764962,0.009793
4,winogrande,2.0,30.0,2.0,2.301533,0.301533,0.697711,0.704815,0.007103
5,hellaswag,2.0,30.0,2.0,2.301533,0.301533,0.534157,0.525891,-0.008265
6,arc_challenge,3.0,75.0,3.0,2.966646,0.033354,0.481229,0.480375,-0.000853
7,arc_easy,3.0,75.0,3.0,2.966646,0.033354,0.795875,0.788300,-0.007576
8,boolq,3.0,75.0,3.0,2.966646,0.033354,0.814373,0.820795,0.006422
9,piqa,3.0,75.0,3.0,2.966646,0.033354,0.791621,0.784548,-0.007073


Saved Common Sense tables to /home/jgryu/workspace/weight_compression/notebooks/exp_results_csv/paired_ttest_qtip_vs_nwc


In [5]:
mmlu_subject_pairs_df = build_mmlu_subject_pairs(qtip_df, nwc_df, pairs_df)
mmlu_by_pair_df = run_mmlu_tests_by_pair(mmlu_subject_pairs_df)

print(f"MMLU p-value rows: {len(mmlu_by_pair_df)}")
print(f"Expected rows from MANUAL_KEY_PAIRS: {len(MANUAL_KEY_PAIRS)}")

print("MMLU paired t-test by manual key pair")
display(mmlu_by_pair_df)

print("Matched MMLU subject-level values")
display(mmlu_subject_pairs_df)

if SAVE_CSV:
    mmlu_by_pair_df.to_csv(OUTPUT_DIR / "mmlu_paired_ttest_by_pair.csv", index=False)
    mmlu_subject_pairs_df.to_csv(OUTPUT_DIR / "mmlu_subject_pairs.csv", index=False)
    print(f"Saved MMLU tables to {OUTPUT_DIR}")


MMLU p-value rows: 5
Expected rows from MANUAL_KEY_PAIRS: 5
MMLU paired t-test by manual key pair


,qtip_key,nwc_key,qtip_bpp,nwc_bpp,n_subjects,qtip_mean,nwc_mean,mean_diff_nwc_minus_qtip,t_stat,p_value
0,2.0,30.0,2.0,2.301533,57,0.514874,0.513656,-0.001218,-0.129060,0.897773
1,3.0,75.0,3.0,2.966646,57,0.605870,0.601505,-0.004365,-1.211854,0.230657
2,4.0,300.0,4.0,3.954827,57,0.632355,0.634562,0.002207,0.676417,0.501560
3,5.0,1000.0,5.0,4.777825,57,0.633312,0.635748,0.002435,0.883398,0.380798
4,6.0,10000.0,6.0,5.981024,57,0.630649,0.636926,0.006277,2.207383,0.031402


Matched MMLU subject-level values


,subject,qtip_key,nwc_key,qtip_bpp,nwc_bpp,abs_bpp_gap,qtip_score,nwc_score,diff_nwc_minus_qtip
0,mmlu_abstract_algebra,2.0,30.0,2.0,2.301533,0.301533,0.290000,0.230000,-0.060000
1,mmlu_anatomy,2.0,30.0,2.0,2.301533,0.301533,0.518519,0.525926,0.007407
2,mmlu_astronomy,2.0,30.0,2.0,2.301533,0.301533,0.486842,0.592105,0.105263
3,mmlu_business_ethics,2.0,30.0,2.0,2.301533,0.301533,0.600000,0.520000,-0.080000
4,mmlu_clinical_knowledge,2.0,30.0,2.0,2.301533,0.301533,0.558491,0.607547,0.049057
...,...,...,...,...,...,...,...,...,...
280,mmlu_security_studies,6.0,10000.0,6.0,5.981024,0.018976,0.730612,0.714286,-0.016327
281,mmlu_sociology,6.0,10000.0,6.0,5.981024,0.018976,0.835821,0.830846,-0.004975
282,mmlu_us_foreign_policy,6.0,10000.0,6.0,5.981024,0.018976,0.860000,0.870000,0.010000
283,mmlu_virology,6.0,10000.0,6.0,5.981024,0.018976,0.524096,0.530120,0.006024


Saved MMLU tables to /home/jgryu/workspace/weight_compression/notebooks/exp_results_csv/paired_ttest_qtip_vs_nwc
